# Advanced: Multi-Strategy Phone Analysis with PAMOLA.CORE

**Goal**: Master all 4 processing strategies for large-scale phone field analysis using operation.execute()

**What you'll learn:**
- **Strategy 1**: Standard processing (baseline performance)
- **Strategy 2**: Chunked processing (memory-efficient for large datasets)
- **Strategy 3**: Dask parallel processing (distributed computing)
- **Strategy 4**: Joblib parallel processing (multi-core optimization)
- Compare performance across processing methods
- Advanced phone validation and parsing
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of phone number formats
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_phone_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.phone import PhoneOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 4 columns)
- Sample records (first 10 rows)
- Column statistics (types, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'phone_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Country codes with different probabilities
    country_codes = ['1', '7', '33', '44', '49', '86', '91', '55', '61', '81']
    country_weights = [0.25, 0.20, 0.15, 0.12, 0.10, 0.08, 0.05, 0.03, 0.01, 0.01]
    
    # Generate phone numbers with various formats
    phones = []
    for i in range(n_records):
        country = np.random.choice(country_codes, p=country_weights)
        
        if country == '1':  # US/Canada
            area = np.random.randint(200, 999)
            exchange = np.random.randint(200, 999)
            number = np.random.randint(1000, 9999)
            phone = f"+{country}-{area}-{exchange}-{number}"
        elif country == '7':  # Russia
            operator = np.random.choice(['495', '499', '812', '926', '950'])
            number = ''.join([str(np.random.randint(0, 9)) for _ in range(7)])
            phone = f"+{country}-{operator}-{number[:3]}-{number[3:5]}-{number[5:]}"
        elif country == '44':  # UK
            area = np.random.choice(['20', '131', '151', '7911'])
            number = ''.join([str(np.random.randint(0, 9)) for _ in range(7)])
            phone = f"+{country}-{area}-{number}"
        else:  # Other countries
            operator = ''.join([str(np.random.randint(1, 9)) for _ in range(2)])
            number = ''.join([str(np.random.randint(0, 9)) for _ in range(8)])
            phone = f"+{country}-{operator}-{number}"
        
        # Add messenger mentions to some phones (20%)
        if np.random.random() < 0.20:
            messenger = np.random.choice(['telegram', 'whatsapp', 'viber', 'signal', 'line'])
            phone = f"{phone}, {messenger}"
        
        phones.append(phone)
    
    # Create DataFrame
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'phone': phones,
        'customer_name': [f'Customer {i}' for i in range(1, n_records + 1)],
        'contact_type': np.random.choice(['Personal', 'Business', 'Support'], n_records)
    })
    
    # Add some null values (3%)
    null_indices = np.random.choice(n_records, int(n_records * 0.03), replace=False)
    df.loc[null_indices, 'phone'] = np.nan
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s})")

print(f"\n💡 Perfect dataset for testing all 4 processing strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field"`: For standard processing (default: "phone")
   - `"strategy2_field"`: For chunked processing (default: "phone")
   - `"strategy3_field"`: For Dask processing (default: "phone")
   - `"strategy4_field"`: For Joblib processing (default: "phone")
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Data type and sample values for each field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset and contain phone strings before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "phone",   # Standard processing
    "strategy2_field": "phone",   # Chunked processing
    "strategy3_field": "phone",   # Dask parallel
    "strategy4_field": "phone"    # Joblib parallel
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    print(f"  ✓ {strategy:20s}: '{field_name}'")
    print(f"    Type: {df[field_name].dtype}")
    print(f"    Non-null: {df[field_name].notna().sum()}")
    print(f"    Sample: {df[field_name].dropna().iloc[0] if len(df[field_name].dropna()) > 0 else 'N/A'}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy phone field analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Standard Processing

**How to use:**
- Baseline processing without optimization
- Single-threaded, in-memory analysis
- Run to execute standard strategy

**What you'll see:**
- Configuration summary
- Progress: validation → parsing → extraction → save
- Completion time and status
- Artifact count (6-8 files expected)

**Key parameters:**
- `min_frequency=1`: Include all values in dictionaries
- `patterns_csv=None`: Use default messenger patterns
- `country_codes=None`: Analyze all countries
- `use_vectorization=False`: No parallel processing
- `use_dask=False`: No distributed computing

**Note:** Provides baseline performance for comparison with optimized strategies

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: STANDARD PROCESSING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Standard",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = PhoneOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    min_frequency=1,
    patterns_csv=None,
    country_codes=None,
    
    # Output configuration
    output_format='json',
    
    # Processing settings (STANDARD - no optimization)
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    chunk_size=10000,
    use_dask=False,
    npartitions=None,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_standard',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_standard' / 'output').glob('*_stats_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    with open(output_files_s1[0], 'r') as f:
        result_data_s1 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid phones: {result_data_s1.get('valid_count', 0):,}")
    print(f"   Null phones: {result_data_s1.get('null_count', 0):,} ({result_data_s1.get('null_percentage', 0):.2f}%)")
    print(f"   Format errors: {result_data_s1.get('format_error_count', 0):,}")
    print(f"   Unique countries: {len(result_data_s1.get('country_codes', {})):,}")

## STRATEGY 2: Chunked Processing

**How to use:**
- Memory-efficient processing for large datasets
- Processes data in manageable chunks
- Run to execute chunked strategy

**What you'll see:**
- Configuration summary
- Progress: validation → chunked processing → save
- Completion time and status
- Artifact count (6-8 files expected)

**Key parameters:**
- `chunk_size=250`: Process 250 rows at a time
- `use_vectorization=False`: Sequential chunk processing
- `use_dask=False`: No distributed computing
- Lower memory footprint than Strategy 1

**Note:** Best for datasets that don't fit in memory or when memory is constrained

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: CHUNKED PROCESSING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Chunked",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = PhoneOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    min_frequency=1,
    patterns_csv=None,
    country_codes=None,
    
    # CHUNKED processing settings
    chunk_size=250,
    use_vectorization=False,
    use_dask=False,
    
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"   Chunk size: {operation_s2.chunk_size} rows")
print(f"   Total chunks: {len(df) // operation_s2.chunk_size + 1}")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_chunked',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_chunked' / 'output').glob('*_stats_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    with open(output_files_s2[0], 'r') as f:
        result_data_s2 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid phones: {result_data_s2.get('valid_count', 0):,}")
    print(f"   Memory efficient: ✓")

## STRATEGY 3: Dask Parallel Processing

**How to use:**
- Distributed computing with Dask
- Parallel processing across multiple partitions
- Run to execute Dask strategy

**What you'll see:**
- Configuration summary
- Progress: validation → parallel processing → save
- Completion time and status
- Artifact count (6-8 files expected)

**Key parameters:**
- `use_dask=True`: Enable Dask distributed computing
- `npartitions=4`: Split data into 4 partitions
- `chunk_size=250`: Chunk size per partition
- Scales to very large datasets

**Note:** Best for very large datasets (10M+ rows) or distributed computing environments

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: DASK PARALLEL PROCESSING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Dask",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = PhoneOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    min_frequency=1,
    patterns_csv=None,
    country_codes=None,
    
    # DASK parallel processing settings
    use_dask=True,
    npartitions=4,
    chunk_size=250,
    use_vectorization=False,
    
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"   Partitions: {operation_s3.npartitions}")
print(f"   Parallel: Dask distributed")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_dask',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_dask' / 'output').glob('*_stats_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    with open(output_files_s3[0], 'r') as f:
        result_data_s3 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid phones: {result_data_s3.get('valid_count', 0):,}")
    print(f"   Distributed: ✓")

## STRATEGY 4: Joblib Parallel Processing

**How to use:**
- Multi-core parallel processing with Joblib
- Optimal for single-machine parallelization
- Run to execute Joblib strategy

**What you'll see:**
- Configuration summary
- Progress: validation → parallel processing → save
- Completion time and status
- Artifact count (6-8 files expected)

**Key parameters:**
- `use_vectorization=True`: Enable parallel processing
- `parallel_processes=4`: Use 4 CPU cores
- `chunk_size=250`: Chunk size per process
- Best single-machine performance

**Note:** Optimal for medium-to-large datasets on multi-core machines (not distributed)

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 4: JOBLIB PARALLEL PROCESSING")
print("=" * 80 + "\n")

tracker_s4 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 4: Joblib",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s4 = PhoneOperation(
    field_name=FIELD_CONFIG["strategy4_field"],
    min_frequency=1,
    patterns_csv=None,
    country_codes=None,
    
    # JOBLIB parallel processing settings
    use_vectorization=True,
    parallel_processes=4,
    chunk_size=250,
    use_dask=False,
    
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 4 configured")
print(f"   Processes: {operation_s4.parallel_processes}")
print(f"   Parallel: Joblib multi-core")
start_time = time.time()

result_s4 = operation_s4.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy4_joblib',
    reporter=task_reporter,
    progress_tracker=tracker_s4,
    **kwargs
)

elapsed_s4 = time.time() - start_time
print(f"\n✅ Strategy 4 completed in {elapsed_s4:.2f}s")

# Load results
output_files_s4 = sorted(
    list((task_dir / 'strategy4_joblib' / 'output').glob('*_stats_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s4:
    with open(output_files_s4[0], 'r') as f:
        result_data_s4 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid phones: {result_data_s4.get('valid_count', 0):,}")
    print(f"   Multi-core: ✓")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Speedup factors vs baseline
- Strategy rankings by performance

**Note:** Fastest strategy depends on dataset size, hardware, and available cores

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Standard): {elapsed_s1:6.2f}s  [baseline]")
print(f"  Strategy 2 (Chunked):  {elapsed_s2:6.2f}s  [{elapsed_s1/elapsed_s2:.2f}x]")
print(f"  Strategy 3 (Dask):     {elapsed_s3:6.2f}s  [{elapsed_s1/elapsed_s3:.2f}x]")
print(f"  Strategy 4 (Joblib):   {elapsed_s4:6.2f}s  [{elapsed_s1/elapsed_s4:.2f}x]")
print(f"  Total:                 {elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4:6.2f}s")

# Find fastest strategy
strategies = [
    ('Standard', elapsed_s1),
    ('Chunked', elapsed_s2),
    ('Dask', elapsed_s3),
    ('Joblib', elapsed_s4)
]
strategies.sort(key=lambda x: x[1])

print(f"\n🏆 Performance Ranking:")
for i, (name, time_val) in enumerate(strategies, 1):
    print(f"  {i}. {name:12s} - {time_val:.2f}s")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Stats JSON**: Phone validation and parsing metrics
2. **Dictionaries**: Country codes, operator codes, messenger mentions
3. **Visualizations**: Distribution charts (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_standard', 'Strategy 1: Standard Processing'),
    ('strategy2_chunked', 'Strategy 2: Chunked Processing'),
    ('strategy3_dask', 'Strategy 3: Dask Parallel'),
    ('strategy4_joblib', 'Strategy 4: Joblib Parallel')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Phone Number Statistics (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir and output_dir.exists():
        stats_files = sorted(
            list(output_dir.glob('*_stats_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if stats_files:
            print(f"\n📞 PHONE NUMBER ANALYSIS: {stats_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(stats_files[0], 'r') as f:
                    stats = json.load(f)
                
                # Basic statistics
                print(f"   Total rows           : {stats.get('total_rows', 0):,}")
                print(f"   Valid phones         : {stats.get('valid_count', 0):,} ({stats.get('valid_percentage', 0):.2f}%)")
                print(f"   Null phones          : {stats.get('null_count', 0):,} ({stats.get('null_percentage', 0):.2f}%)")
                print(f"   Format errors        : {stats.get('format_error_count', 0):,}")
                
                # Normalization success
                if 'normalization_success_count' in stats:
                    print(f"\n   🔧 Normalization:")
                    print(f"      Success count     : {stats.get('normalization_success_count', 0):,}")
                    print(f"      Success rate      : {stats.get('normalization_success_percentage', 0):.2f}%")
                
                # Additional features
                print(f"\n   📋 Additional Features:")
                print(f"      Has comments      : {stats.get('has_comment_count', 0):,}")
                print(f"      Has extensions    : {stats.get('has_extension_count', 0):,}")
                
                # Country codes distribution
                if 'country_codes' in stats and stats['country_codes']:
                    country_codes = stats['country_codes']
                    total_phones = sum(country_codes.values())
                    
                    print(f"\n   🌍 Country Codes ({len(country_codes)} unique):")
                    
                    # Sort by count descending
                    sorted_countries = sorted(
                        country_codes.items(), 
                        key=lambda x: x[1], 
                        reverse=True
                    )
                    
                    for code, count in sorted_countries[:10]:  # Show top 10
                        pct = (count / total_phones * 100) if total_phones > 0 else 0
                        print(f"      +{code:3s}             : {count:4,} ({pct:5.2f}%)")
                    
                    if len(country_codes) > 10:
                        print(f"      ... and {len(country_codes) - 10} more countries")
                
                # Operator codes distribution
                if 'operator_codes' in stats and stats['operator_codes']:
                    operator_codes = stats['operator_codes']
                    total_operators = sum(operator_codes.values())
                    
                    print(f"\n   📡 Operator Codes ({len(operator_codes)} unique):")
                    
                    # Sort by count descending
                    sorted_operators = sorted(
                        operator_codes.items(), 
                        key=lambda x: x[1], 
                        reverse=True
                    )
                    
                    for op_code, count in sorted_operators[:10]:  # Show top 10
                        pct = (count / total_operators * 100) if total_operators > 0 else 0
                        print(f"      {op_code:15s}: {count:3,} ({pct:5.2f}%)")
                    
                    if len(operator_codes) > 10:
                        print(f"      ... and {len(operator_codes) - 10} more operators")
                
                # Messenger mentions
                if 'messenger_mentions' in stats:
                    messenger_mentions = stats['messenger_mentions']
                    total_mentions = sum(messenger_mentions.values())
                    
                    if total_mentions > 0:
                        print(f"\n   💬 Messenger Mentions ({total_mentions} total):")
                        
                        # Sort by count descending
                        sorted_messengers = sorted(
                            messenger_mentions.items(), 
                            key=lambda x: x[1], 
                            reverse=True
                        )
                        
                        for messenger, count in sorted_messengers:
                            if count > 0:  # Only show messengers with mentions
                                pct = (count / total_mentions * 100) if total_mentions > 0 else 0
                                messenger_display = messenger.capitalize()
                                print(f"      {messenger_display:15s}: {count:3,} ({pct:5.2f}%)")
                
                # Format error examples
                if 'format_error_examples' in stats and stats['format_error_examples']:
                    print(f"\n   ⚠️  Format Error Examples:")
                    for idx, example in enumerate(stats['format_error_examples'][:5], 1):
                        print(f"      {idx}. {example}")
                    
                    if len(stats['format_error_examples']) > 5:
                        print(f"      ... and {len(stats['format_error_examples']) - 5} more")
                
                # Extension examples
                if 'extension_examples' in stats and stats['extension_examples']:
                    print(f"\n   📞 Extension Examples:")
                    for idx, example in enumerate(stats['extension_examples'][:5], 1):
                        print(f"      {idx}. {example}")
                    
                    if len(stats['extension_examples']) > 5:
                        print(f"      ... and {len(stats['extension_examples']) - 5} more")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
                import traceback
                traceback.print_exc()
    
    # 2. Dictionaries Overview (from dictionaries/)
    dict_dir = strategy_dir / 'dictionaries'
    if dict_dir and dict_dir.exists():
        dict_files = sorted(
            list(dict_dir.glob('*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if dict_files:
            print(f"\n📚 DICTIONARIES: {len(dict_files)} files")
            print("   " + "-" * 60)
            
            for df_file in dict_files[:5]:  # Show first 5
                # Try to read and show basic info
                try:
                    df = pd.read_csv(df_file, nrows=0)  # Read only header
                    row_count = sum(1 for _ in open(df_file)) - 1  # Count lines minus header
                    print(f"   • {df_file.name:50s} ({row_count:,} rows)")
                except:
                    print(f"   • {df_file.name}")
            
            if len(dict_files) > 5:
                print(f"   ... and {len(dict_files) - 5} more dictionaries")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Statistical Consistency Check

**How to use:**
- Run AFTER all strategies complete
- Verifies all strategies produce identical results

**What you'll see:**
- Valid phone counts from each strategy
- Country code counts comparison
- Consistency verification (all should match)
- Any discrepancies highlighted

**Note:** All strategies should produce identical analysis results despite different processing methods

In [ ]:
print("\n" + "=" * 80)
print("🔬 STATISTICAL CONSISTENCY CHECK")
print("=" * 80 + "\n")

# Collect statistics from all strategies
all_stats = []
strategy_names = ['Standard', 'Chunked', 'Dask', 'Joblib']

for i, data in enumerate([result_data_s1, result_data_s2, result_data_s3, result_data_s4], 1):
    all_stats.append({
        'strategy': strategy_names[i-1],
        'valid': data.get('valid_count', 0),
        'null': data.get('null_count', 0),
        'errors': data.get('format_error_count', 0),
        'countries': len(data.get('country_codes', {}))
    })

print("📊 Analysis Results Comparison:")
print(f"\n{'Strategy':<12} {'Valid':>8} {'Null':>8} {'Errors':>8} {'Countries':>10}")
print("-" * 50)
for stat in all_stats:
    print(f"{stat['strategy']:<12} {stat['valid']:>8,} {stat['null']:>8,} {stat['errors']:>8,} {stat['countries']:>10}")

# Check consistency
valid_counts = [s['valid'] for s in all_stats]
country_counts = [s['countries'] for s in all_stats]

valid_diff = max(valid_counts) - min(valid_counts)
country_diff = max(country_counts) - min(country_counts)

print(f"\n✅ Consistency Check:")
print(f"   Valid count variance: {valid_diff}")
print(f"   Country count variance: {country_diff}")

if valid_diff == 0 and country_diff == 0:
    print(f"   Status: ✓ All strategies produce identical results")
else:
    print(f"   Status: ⚠️  Some variance detected")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports analysis summary for each strategy

**What you'll see (per strategy):**
- Export confirmation with directory path
- Key statistics summary
- Processing method and performance

**Output structure:**
```
advanced_output/
├── strategy1_standard/
│   ├── output/*.json
│   └── dictionaries/*.csv
├── strategy2_chunked/
│   ├── output/*.json
│   └── dictionaries/*.csv
├── strategy3_dask/
│   ├── output/*.json
│   └── dictionaries/*.csv
└── strategy4_joblib/
    ├── output/*.json
    └── dictionaries/*.csv
```

**Note:** Each strategy folder contains complete analysis results with metrics and visualizations

In [ ]:
print("=" * 80)
print("📦 EXPORT SUMMARY")
print("=" * 80)

print(f"\n📂 All files saved to: {task_dir}")

strategies_info = [
    ('strategy1_standard', 'Standard Processing', elapsed_s1, result_data_s1),
    ('strategy2_chunked', 'Chunked Processing', elapsed_s2, result_data_s2),
    ('strategy3_dask', 'Dask Parallel', elapsed_s3, result_data_s3),
    ('strategy4_joblib', 'Joblib Parallel', elapsed_s4, result_data_s4)
]

print(f"\n📁 Strategy Summaries:")
for dir_name, strategy_name, elapsed, result_data in strategies_info:
    print(f"\n  {strategy_name}:")
    print(f"    Directory: {dir_name}/")
    print(f"    Processing time: {elapsed:.2f}s")
    print(f"    Valid phones: {result_data.get('valid_count', 0):,}")
    print(f"    Unique countries: {len(result_data.get('country_codes', {})):,}")
    print(f"    Messenger mentions: {result_data.get('has_comment_count', 0):,}")

total_time = sum([elapsed_s1, elapsed_s2, elapsed_s3, elapsed_s4])
print(f"\n⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 4 processing strategies implemented and compared
- ✅ Performance benchmarking completed
- ✅ Statistical consistency verified across strategies
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Standard processing: Simple, reliable baseline
- Chunked processing: Memory-efficient for large datasets
- Dask parallel: Best for distributed/very large datasets
- Joblib parallel: Optimal single-machine multi-core performance

**Strategy recommendations:**
- **Use Standard** when: Dataset < 100K rows, simple analysis needed
- **Use Chunked** when: Memory constrained, dataset > 1M rows
- **Use Dask** when: Dataset > 10M rows, distributed environment available
- **Use Joblib** when: Dataset 100K-10M rows, multi-core machine available

**Next steps:**
- Test with your own phone datasets
- Tune parameters (chunk_size, npartitions, parallel_processes)
- Add custom messenger patterns
- Deploy to production environment
- Monitor performance metrics

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)